<a href="https://colab.research.google.com/github/cerr/pycerr-notebooks/blob/main/09_functional_imaging/compute_suv_from_dro.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Compute PET SUV from the IBSI-SUV digital reference objects

# Introduction

The standardized uptake value (SUV) is the most widely used semi-quantitative
measure in PET, but converting stored DICOM voxel values to SUV depends on a
long chain of metadata: the image units, the decay-correction reference time,
the administered dose, and the normalization the scanner applied. Different
readers disagree, which makes SUVs hard to compare across studies.

The [IBSI-SUV](https://github.com/oncoray/suv_computation) project publishes a
set of digital reference objects (DROs) to standardize this conversion. Every
DRO encodes the **same synthetic phantom** but varies how the values and
metadata are stored: units (`BQML`, `CNTS`, `GML`, `CM2ML`), decay correction
(`START`, `ADMIN`, `NONE`), vendor private tags (GE, Siemens, Philips), dose
units, administration dates, and the Enhanced PET Image IOD.

Because the underlying phantom is identical, a correct implementation returns
the **same SUVbw statistics for every DRO**:

| region | SUVbw |
|---|---|
| cold sphere (minimum) | 0.2 |
| background (median) | 1.0 |
| hot sphere (maximum) | 4.0 |

This notebook downloads the DROs directly from GitHub, computes SUV with
pyCERR for each one, and reports the minimum, median and maximum inside the
tumour mask so the results can be checked against those reference values.

### Requirements
* Python>=3.8
* Network access (the DROs and pyCERR are both fetched from GitHub)

### I/O
* Input: DICOM PET series and RTSTRUCT tumour masks, cloned from the public
  IBSI-SUV repository. No local paths and no patient data are involved: the
  DROs are synthetic phantoms.
* Output: a table of DRO name, minimum, median and maximum SUVbw.

### References
* Vácha, M., Zwanenburg, A., et al. *Standardizing SUV computation (IBSI-SUV)*.
  https://oncoray.github.io/suv_computation/suv.html
* Boellaard, R., et al. (2015) *FDG PET/CT: EANM procedure guidelines for tumour
  imaging: version 2.0*. Eur J Nucl Med Mol Imaging 42:328-354.
* Kim, C.K., et al. (1994) *Expression of the standardized uptake value (SUV)*.
  J Nucl Med 35:164-167.

-----

## License

By downloading the software you are agreeing to the following terms and conditions as well as to the Terms of Use of CERR software.
```
THE SOFTWARE IS PROVIDED "AS IS" AND CERR DEVELOPMENT TEAM AND ITS COLLABORATORS DO NOT MAKE ANY WARRANTY, EXPRESS OR IMPLIED, INCLUDING BUT NOT LIMITED TO WARRANTIES OF MERCHANTABILITY AND FITNESS FOR A PARTICULAR PURPOSE, NOR DO THEY ASSUME ANY LIABILITY OR RESPONSIBILITY FOR THE USE OF THIS SOFTWARE.

This software is for research purposes only and has not been approved for clinical use.

Software has not been reviewed or approved by the Food and Drug Administration, and is for non-clinical, IRB-approved Research Use Only. In no event shall data or images generated through the use of the Software be used in the provision of patient care.

YOU MAY NOT DISTRIBUTE COPIES of this software, or copies of software derived from this software, to others outside your organization without specific prior written permission from the CERR development team except where noted for specific software products.

All Technology and technical data delivered under this Agreement are subject to US export control laws and may be subject to export or import regulations in other countries. You agree to comply strictly with all such laws and regulations and acknowledge that you have the responsibility to obtain such licenses to export, re-export, or import as may be required after delivery to you.
```
--------

## Install pyCERR

In [ ]:
%%capture
!pip install "pyCERR @ git+https://github.com/cerr/pyCERR.git@testing"

## Download the digital reference objects

The DROs are pulled straight from the public IBSI-SUV repository with a shallow
clone (about 25 MB). Nothing is read from a local disk, so this notebook runs
unchanged on Colab or any fresh environment.

In [2]:
import os
import subprocess

DRO_REPO = 'https://github.com/oncoray/suv_computation.git'
CLONE_DIR = 'suv_computation'

if not os.path.isdir(CLONE_DIR):
    subprocess.run(['git', 'clone', '--depth', '1', DRO_REPO, CLONE_DIR],
                   check=True)

DRO_ROOT = os.path.join(CLONE_DIR, 'DRO')
print('DROs available at:', DRO_ROOT)

DROs available at: suv_computation\DRO


## Identify the available DROs

The DRO list is discovered from the repository rather than hard-coded, so the
notebook keeps working as the IBSI-SUV collection grows.

The collection contains two groups:

* **Reference DROs** — complete metadata; SUV must be computable and must match
  0.2 / 1.0 / 4.0.
* **`error` DROs** — deliberately missing or invalid attributes (no patient
  weight, no radiopharmaceutical time, unusable units). Here SUV must *not* be
  computed; the correct behaviour is to refuse and warn. These are reported
  separately at the end.

In [3]:
referenceDROs = sorted(d for d in os.listdir(DRO_ROOT)
                       if d.startswith('DRO_') and 'error' not in d)
errorDROs = sorted(d for d in os.listdir(DRO_ROOT) if d.startswith('DRO_error'))

print(f'{len(referenceDROs)} reference DROs, {len(errorDROs)} error DROs')

43 reference DROs, 15 error DROs


## Compute SUV for each DRO

`loadDcmDir` imports the PET series and the RTSTRUCT, converts the stored voxel
values to SUV, and returns a `PlanC` object. The SUV normalization is selected
with the `suvType` option; `'BW'` (body weight) is the default and is what the
reference values above correspond to.

Note that no DRO-specific handling appears below — the same three lines are
applied to every DRO regardless of its units, decay correction or vendor. All
of the variation is absorbed by pyCERR during import, which is the property the
DROs are designed to test.

In [4]:
import warnings
import numpy as np
from cerr import plan_container as pc
from cerr.contour import rasterseg as rs


def suvStatsForDRO(droDir, suvType='BW'):
    """Return (min, median, max) SUV inside the tumour mask of one DRO."""
    planC = pc.loadDcmDir(droDir, opts={'suvType': suvType})
    maskM = rs.getStrMask(0, planC)          # structure 0 is the DRO mask
    suvM = planC.scan[0].getScanArray()      # SUV, already converted on import
    suvV = suvM[maskM]
    return np.min(suvV), np.median(suvV), np.max(suvV)

### Run the conversion

`loadDcmDir` prints a short per-series summary as it imports; it is suppressed
here so the results table stays readable.

In [5]:
import contextlib
import io

results = []
for droName in referenceDROs:
    droDir = os.path.join(DRO_ROOT, droName)
    try:
        # Silence the import banner; surface only genuine warnings.
        with contextlib.redirect_stdout(io.StringIO()):
            with warnings.catch_warnings():
                warnings.simplefilter('ignore')
                suvMin, suvMedian, suvMax = suvStatsForDRO(droDir)
        results.append((droName, suvMin, suvMedian, suvMax))
    except Exception as err:
        results.append((droName, None, None, None))
        print(f'{droName}: import failed - {type(err).__name__}: {err}')

print(f'Computed SUV for {sum(r[1] is not None for r in results)} '
      f'of {len(referenceDROs)} DROs')

Computed SUV for 43 of 43 DROs


## Results

Minimum, median and maximum SUVbw inside the tumour mask of each DRO. The
`check` column marks agreement with the expected 0.2 / 1.0 / 4.0 within 2%.

In [6]:
EXPECTED = (0.2, 1.0, 4.0)
TOLERANCE = 0.02


def matchesReference(stats):
    return all(abs(value - expected) <= TOLERANCE * expected
               for value, expected in zip(stats, EXPECTED))


header = f"{'DRO':<14}{'min':>9}{'median':>10}{'max':>9}   check"
print(header)
print('-' * len(header))

nPass = 0
for droName, suvMin, suvMedian, suvMax in results:
    if suvMin is None:
        print(f'{droName:<14}{"-":>9}{"-":>10}{"-":>9}   not computed')
        continue
    ok = matchesReference((suvMin, suvMedian, suvMax))
    nPass += ok
    print(f'{droName:<14}{suvMin:9.3f}{suvMedian:10.3f}{suvMax:9.3f}'
          f'   {"OK" if ok else "MISMATCH"}')

print('-' * len(header))
print(f'expected      {EXPECTED[0]:9.3f}{EXPECTED[1]:10.3f}{EXPECTED[2]:9.3f}')
print(f'\n{nPass} of {len(results)} DROs match the reference values.')

DRO                 min    median      max   check
--------------------------------------------------
DRO_0_0           0.200     1.000    4.000   OK
DRO_1_0           0.200     1.000    4.000   OK
DRO_2_0           0.200     1.000    4.000   OK
DRO_2_1_0         0.200     1.000    4.000   OK
DRO_2_1_1         0.200     1.000    4.000   OK
DRO_2_1_2         0.200     1.000    4.000   OK
DRO_2_2_0         0.199     1.000    4.000   OK
DRO_2_2_1         0.198     0.999    4.000   OK
DRO_2_2_2         0.200     0.998    4.000   OK
DRO_2_3           0.200     1.000    4.000   OK
DRO_2_4           0.200     1.000    4.000   OK
DRO_2_5           0.200     1.000    4.000   OK
DRO_2_6_0         0.200     1.000    4.000   OK
DRO_2_6_1         0.200     1.000    4.000   OK
DRO_2_6_2         0.199     1.000    4.000   OK
DRO_3_0           0.200     1.000    4.000   OK
DRO_3_1           0.200     1.000    4.000   OK
DRO_3_2_0         0.200     1.000    4.000   OK
DRO_3_2_1         0.200     1.000 

## DROs where SUV should not be computed

The `error` DROs are missing attributes that SUV computation requires, or carry
values that cannot be interpreted. Returning a number for these would be worse
than returning nothing, so pyCERR warns and leaves the scan unconverted. The
warning text is shown below.

In [7]:
header = f"{'DRO':<16}reason SUV was not computed"
print(header)
print('-' * 78)

for droName in errorDROs:
    droDir = os.path.join(DRO_ROOT, droName)
    with contextlib.redirect_stdout(io.StringIO()):
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            try:
                pc.loadDcmDir(droDir)
                messages = [str(w.message) for w in caught]
            except Exception as err:
                messages = [f'{type(err).__name__}: {err}']
    reason = messages[0] if messages else 'no warning raised'
    print(f'{droName:<16}{reason[:62]}')

DRO             reason SUV was not computed
------------------------------------------------------------------------------


DRO_error_2_0   Patient's Weight is missing or non-positive; SUV cannot be com


DRO_error_2_1   Patient's Weight is missing or non-positive; SUV cannot be com


DRO_error_2_2   Patient attributes required to convert stored IBW values to SU


DRO_error_2_3   Patient attributes required to convert stored IBW values to SU


...

DRO_error_3_2   Actual Frame Duration is required to back-compute the scan sta


DRO_error_4_0   Radiopharmaceutical administration datetime is unavailable.


DRO_error_4_1   Radiopharmaceutical administration date is inconsistent with t


DRO_error_4_2   Radiopharmaceutical administration date is inconsistent with t


DRO_error_5_0   Radiopharmaceutical administration date is inconsistent with t


## Other SUV normalizations

SUVbw is the default and the most widely reported, but the DICOM standard also
allows normalization by lean body mass, ideal body weight and body surface
area. pyCERR converts between them: whatever normalization the scanner applied,
the requested one is produced.

`'AS_STORED'` is also accepted and keeps the scanner's own normalization. It is
useful for inspecting a file as-is, but because the result depends on the
acquisition protocol rather than on the request, it should not be used to pool
measurements across a cohort.

In [8]:
suvTypes = ['BW', 'LBM', 'LBMJAMES128', 'LBMJANMA', 'IBW', 'BSA']
droDir = os.path.join(DRO_ROOT, 'DRO_0_0')

header = f"{'suvType':<14}{'min':>9}{'median':>10}{'max':>9}"
print(header)
print('-' * len(header))
for suvType in suvTypes:
    with contextlib.redirect_stdout(io.StringIO()):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            stats = suvStatsForDRO(droDir, suvType=suvType)
    print(f'{suvType:<14}{stats[0]:9.3f}{stats[1]:10.3f}{stats[2]:9.3f}')

suvType             min    median      max
------------------------------------------


BW                0.200     1.000    4.000


LBM               0.156     0.779    3.115


LBMJAMES128       0.154     0.770    3.078


LBMJANMA          0.144     0.722    2.887


IBW               0.198     0.992    3.966


BSA               0.053     0.264    1.056
